# TUGAS MANDIRI — Pertemuan 5
## Dashboard Performa Cabang Toko: Join, Window Function & Spark SQL

| | |
|---|---|
| **Nama** | Hilmi Mufid |
| **NIM** | 2505060046 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 5 |

---

### Konteks

Manajemen platform e-commerce (skenario lanjutan dari Pertemuan 2-4) meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi yang sudah tersimpan di HDFS sejak Pertemuan 3-4 dengan data referensi target penjualan tiap cabang.

Notebook ini menyiapkan analisis tersebut menggunakan kombinasi **join, window function, dan Spark SQL**:
- **Bagian A & B** menggunakan **DataFrame API** (sesuai ketentuan).
- **Bagian C** menggunakan **Spark SQL murni** lewat `spark.sql(...)`.
- **Bagian D** berisi kesimpulan berbasis angka hasil analisis A dan B.

> **Catatan lingkungan & path:** Dijalankan di laptop Ubuntu (dual-boot), Spark/PySpark 3.5.9, Hadoop 3.4.3. Mengikuti konvensi direktori kerja pribadi yang dipakai sejak Pertemuan 3 (`/home/mufid/praktikum-bigdata0046`), maka `/user/mahasiswa/tugas5` pada instruksi soal disesuaikan menjadi `/home/mufid/praktikum-bigdata0046/tugas5`.

---
## Persiapan: Dataset & SparkSession

Dua tabel disiapkan sesuai instruksi modul:
1. **`df_transaksi`** — tabel fakta, dibaca dari CSV yang diunggah ke HDFS.
2. **`df_target`** — tabel referensi (*dimension table*) target & PIC per cabang, dibuat langsung sebagai DataFrame karena berukuran kecil dan jarang berubah.

In [1]:
# Membuat dataset dan mengunggah tabel transaksi ke HDFS
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /home/mufid/praktikum-bigdata0046/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /home/mufid/praktikum-bigdata0046/tugas5/
print("Dataset siap dan sudah diunggah ke HDFS.")

Dataset siap dan sudah diunggah ke HDFS.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number, round as spark_round
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


### Langkah 1 & 2 — Membaca `df_transaksi` dari HDFS dan membuat `df_target`

`df_transaksi` dibaca **langsung dari HDFS** (prefix `hdfs://localhost:9000/...`), lalu ditambahkan kolom `pendapatan`. `df_target` dibuat dari dictionary `data_target_cabang` menggunakan `spark.createDataFrame()`.

In [4]:
HDFS_PATH = "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas5/transaksi_tugas5.csv"

# Langkah 1: baca dari HDFS + tambahkan kolom pendapatan
df_transaksi = spark.read.csv(HDFS_PATH, header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

print("df_transaksi — total baris:", df_transaksi.count())
df_transaksi.show(5)

# Langkah 2: buat df_target dari dictionary tabel referensi
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("df_target (tabel referensi):")
df_target.show()

df_transaksi — total baris: 500
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

df_target (tabel referensi):


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



---
## A. Join & Perbandingan Target *(bobot 25%)*

**Penjelasan:** Mengikuti pola kerja yang dipelajari di modul — **agregasi dulu, baru join**. Total `pendapatan` diringkas per `kota` terlebih dahulu (menghasilkan 5 baris saja), baru digabungkan dengan `df_target`. Pola ini jauh lebih efisien dibanding join dulu baru agregasi, karena jumlah baris yang perlu di-join menjadi sangat sedikit (5 baris kota, bukan 500 baris transaksi mentah).

Kolom `pencapaian_persen` dihitung sebagai `total_pendapatan / target_bulanan x 100`.

In [5]:
# Langkah 1: ringkas total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: join dengan tabel target (inner join — semua kota ada di kedua tabel)
hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner")

# Langkah 3: hitung persentase pencapaian terhadap target
hasil_a = hasil_a.withColumn(
    "pencapaian_persen",
    spark_round(col("total_pendapatan") / col("target_bulanan") * 100, 2)
)

# Urutkan dari pencapaian tertinggi
hasil_a = hasil_a.select(
    "kota", "pic_cabang", "total_pendapatan", "target_bulanan", "pencapaian_persen"
).orderBy(col("pencapaian_persen").desc())

hasil_a.show()

+----------+----------+----------------+--------------+-----------------+
|      kota|pic_cabang|total_pendapatan|target_bulanan|pencapaian_persen|
+----------+----------+----------------+--------------+-----------------+
| Purworejo|     Fitri|        45650000|      30000000|           152.17|
|      Solo|      Bayu|        33475000|      40000000|            83.69|
|Yogyakarta|      Joko|        47275000|      60000000|            78.79|
|  Magelang|      Rani|        31650000|      45000000|            70.33|
|  Semarang|      Sari|        38175000|      55000000|            69.41|
+----------+----------+----------------+--------------+-----------------+



**Hasil bagian A:**

| kota | pic_cabang | total_pendapatan | target_bulanan | pencapaian_persen |
|---|---|---|---|---|
| Purworejo | Fitri | 45.650.000 | 30.000.000 | **152,17%** |
| Solo | Bayu | 33.475.000 | 40.000.000 | 83,69% |
| Yogyakarta | Joko | 47.275.000 | 60.000.000 | 78,79% |
| Magelang | Rani | 31.650.000 | 45.000.000 | 70,33% |
| Semarang | Sari | 38.175.000 | 55.000.000 | **69,41%** |

Hanya cabang **Purworejo** yang berhasil melampaui target; empat cabang lainnya masih di bawah 100%.

---
## B. Window Function — Kategori Terlaris per Kota *(bobot 25%)*

**Penjelasan:** Karena yang dicari adalah kategori terlaris (bukan transaksi tunggal terbesar), pendapatan diringkas dulu per kombinasi `kota` + `kategori`. Setelah itu window function diterapkan:
- `partitionBy("kota")` — perankingan dihitung **di dalam tiap kota** secara terpisah.
- `orderBy(col("total_pendapatan").desc())` — diurutkan dari pendapatan tertinggi.
- `row_number()` — sesuai instruksi soal, dipakai agar hasilnya dijamin **tepat 1 baris per kota** meskipun ada nilai yang seri (berbeda dengan `rank()` yang bisa mengembalikan lebih dari satu baris untuk peringkat 1 jika terjadi seri).

In [6]:
# Langkah 1: ringkas pendapatan per kombinasi kota + kategori
pendapatan_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: definisikan window — dipartisi per kota, diurutkan dari pendapatan tertinggi
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Langkah 3: beri nomor urut, lalu ambil hanya peringkat 1 (kategori terlaris) tiap kota
hasil_b = pendapatan_kota_kategori \
    .withColumn("peringkat", row_number().over(window_kota)) \
    .filter(col("peringkat") == 1) \
    .select("kota", "kategori", "total_pendapatan") \
    .orderBy(col("total_pendapatan").desc())

hasil_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|Yogyakarta|             Fashion|        13325000|
|  Semarang|        Rumah Tangga|        11125000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|      Solo|Kesehatan & Kecan...|         8425000|
|  Magelang|Kesehatan & Kecan...|         7275000|
+----------+--------------------+----------------+



**Hasil bagian B kategori terlaris di setiap cabang:**

| kota | kategori terlaris | total_pendapatan |
|---|---|---|
| Yogyakarta | Fashion | 13.325.000 |
| Semarang | Rumah Tangga | 11.125.000 |
| Purworejo | Kesehatan & Kecantikan | 10.075.000 |
| Solo | Kesehatan & Kecantikan | 8.425.000 |
| Magelang | Kesehatan & Kecantikan | 7.275.000 |

Terlihat **Kesehatan & Kecantikan** menjadi kategori terlaris di tiga dari lima cabang (Purworejo, Solo, Magelang).

---
## C. Spark SQL *(bobot 25%)*

**Penjelasan:** Kedua DataFrame didaftarkan sebagai *temporary view* terlebih dahulu, lalu ditulis **satu kueri SQL murni** (bukan DataFrame API) yang men-join kedua view, menghitung `COUNT` jumlah transaksi per kota, dan mengurutkannya dari yang terbanyak.

In [7]:
# Mendaftarkan kedua DataFrame sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Temporary view berhasil didaftarkan: 'transaksi' dan 'target'")

Temporary view berhasil didaftarkan: 'transaksi' dan 'target'


In [8]:
# Satu kueri SQL murni — menampilkan kota, pic_cabang, dan jumlah transaksi
hasil_c = spark.sql("""
    SELECT t.kota,
           g.pic_cabang,
           COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target g ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_c.show()

[Stage 20:==================================================>       (7 + 1) / 8]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**Hasil bagian C:**

| kota | pic_cabang | jumlah_transaksi |
|---|---|---|
| Purworejo | Fitri | 116 |
| Yogyakarta | Joko | 110 |
| Solo | Bayu | 95 |
| Semarang | Sari | 93 |
| Magelang | Rani | 86 |

Total keseluruhan 500 transaksi, tersebar cukup merata namun Purworejo memiliki jumlah transaksi terbanyak.

---
## D. Kesimpulan *(bobot 25%)*

Berdasarkan hasil analisis, cabang Purworejo yang dikoordinasikan oleh PIC Fitri menunjukkan kinerja operasional paling optimal dengan tingkat ketercapaian 152,17% (surplus Rp15,65 juta), didukung oleh volume transaksi tertinggi (116 transaksi) yang didominasi kategori Kesehatan & Kecantikan, meskipun capaian ini turut diuntungkan oleh beban target terendah sebesar Rp30 juta. Sebaliknya, cabang Semarang di bawah PIC Sari serta cabang Magelang memerlukan evaluasi mendalam karena mencatatkan performa paling rentan; khususnya Semarang yang mengalami defisit terbesar senilai Rp16,83 juta dengan tingkat ketercapaian hanya 69,41%. Merespons disparitas kinerja ini, pihak manajemen direkomendasikan untuk meninjau ulang proporsi target operasional agar lebih representatif terhadap potensi pasar masing-masing wilayah demi objektivitas penilaian, serta mengimplementasikan strategi lintas cabang dengan mereplikasi taktik penjualan kategori Kesehatan & Kecantikan dari Purworejo ke Magelang dan Solo guna memaksimalkan pendapatan secara komprehensif.

> **Catatan:** Angka-angka pada kesimpulan di atas mengacu pada dataset dengan `np.random.seed(55)`. Karena seed-nya tetap, hasilnya akan sama setiap kali notebook dijalankan ulang namun tetap disarankan mencocokkan kembali dengan output aktual setelah **Run All**.

---
## Menutup SparkSession

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
